<a href="https://colab.research.google.com/github/mbuguawakahu/CNS4107-Network-Automation-Lab/blob/main/Network_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Network Automation Lab: Device Configuration with Netmiko
**Author:** Joseph Wakahu

## 1. Environment Setup
To begin this network automation lab, we must first install `netmiko`, a multi-vendor library that simplifies SSH connections to network devices.


In [ ]:
# Install the necessary Python library
!pip install netmiko

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.9/262.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.4/642.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.9/223.9 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.3 MB/s eta 0:00:00


## 2. Defining Device Credentials
Initially, the lab targeted the standard public Always-On sandbox (`sandbox-iosxe-latest-1.cisco.com`). However, due to security updates requiring dynamic credentials, I provisioned a dedicated **Catalyst 8000 Always-On Sandbox** via the Cisco DevNet portal. This provides a dynamic host, username, and secure password to bypass the authentication failures of the static sandbox.

In [ ]:
from netmiko import ConnectHandler

# Device details for your active Cisco DevNet Sandbox
device = {
    'device_type': 'cisco_ios',
    'host': 'devnetsandboxiosxec8k.cisco.com',
    'username': 'joseph.wakahu',
    'password': 'lBv4FQ4f-9-K', # Your dynamic password
    'port': 22,
}

print(f"Targeting: {device['host']}")

Targeting: devnetsandboxiosxec8k.cisco.com


## 3. Retrieving Information (Read Operation)
With the credentials securely defined, we use `ConnectHandler` to establish an SSH session. We then execute the `show ip int brief` command to retrieve the current statuses of all interfaces on the router.

In [ ]:
try:
    print("Connecting to the device...")
    net_connect = ConnectHandler(**device)

    print("\n--- Interface Status ---")
    # Run a show command
    output = net_connect.send_command('show ip int brief')
    print(output)

    # Disconnect gracefully
    net_connect.disconnect()
    print("\nDisconnected successfully.")

except Exception as e:
    print(f"Error connecting to the device: {e}")

Connecting to the device...

--- Interface Status ---
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet1       10.10.20.148    YES NVRAM  up                    up      
GigabitEthernet2       unassigned      YES NVRAM  administratively down down    
GigabitEthernet3       unassigned      YES NVRAM  administratively down down    
Loopback0              10.10.10.1      YES manual up                    up      
Loopback396            192.168.3.96    YES other  up                    up      
Loopback566            192.192.192.192 YES other  up                    up      
Loopback670            10.255.255.1    YES other  up                    up      
Vlan2                  10.0.0.1        YES manual down                  down    
Vlan3                  10.0.1.1        YES manual down                  down    
Vlan4                  10.0.2.1        YES manual down                  down    
Vlan5                  10.0.3.1        YES manual down 

## 4. Pushing Configurations (Write Operation)
Next, we automate a configuration change. The script connects to the router and uses `send_config_set()` to create a new virtual interface (`Loopback99`), assign it a description, and allocate an IP address. Finally, it runs a verification command to ensure the interface is UP before gracefully disconnecting.

In [ ]:
try:
    print("Re-establishing connection for config change...")
    # Using the exact same 'device' dictionary from earlier
    net_connect = ConnectHandler(**device)

    # The commands we want to push to configure the router
    config_commands = [
        'interface Loopback99',
        'description Configured via Python Netmiko',
        'ip address 192.168.99.1 255.255.255.255'
    ]

    print("Sending configuration commands...")
    output = net_connect.send_config_set(config_commands)
    print(output)

    print("\n--- Verifying the change ---")
    # Verify by checking if Loopback99 shows up in the interface brief
    verification = net_connect.send_command("show ip interface brief | include Loopback99")
    print(verification)

    # Disconnect gracefully
    net_connect.disconnect()
    print("\nDisconnected successfully.")

except Exception as e:
    print(f"Error during configuration: {e}")

Re-establishing connection for config change...
Sending configuration commands...
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
devnetsandboxiosxec8(config)#interface Loopback99
devnetsandboxiosxec8(config-if)#description Configured via Python Netmiko
devnetsandboxiosxec8(config-if)#ip address 192.168.99.1 255.255.255.255
devnetsandboxiosxec8(config-if)#end
devnetsandboxiosxec8k.cisco.com#

--- Verifying the change ---
Loopback99             192.168.99.1    YES manual up                    up      

Disconnected successfully.
